# SafeCrop: run the CLIP experiments on a free Colab GPU

Use this when your own computer is too slow (E1 noisy copies, E3 fine-tuning). It works in two ways:

* **In the browser**: on [colab.research.google.com](https://colab.research.google.com) choose **File → Upload notebook** and pick this file, then **Runtime → Change runtime type → T4 GPU**.
* **In VS Code** (Colab extension): open this notebook, click **Select Kernel → Colab → New Colab Server → GPU (T4)**.

**Before the first run**, upload two files into your Google Drive folder. Its ID goes in `DRIVE_FOLDER_ID` in the settings cell: the part after `/folders/` in the folder's link.

1. `safecrop_code.zip`: made on your computer with `python scripts/pack_for_colab.py` (it is in `dist/`). Upload a new one whenever the code changes.
2. `safecrop_test_images_840.zip`: the test images and the PlantVillage extras (in `dist/`, or the zip you unzipped into `data/`).

Then **Runtime → Run all**. When Colab asks, sign in with the Google account that can open the folder and click **Allow**. The results come back to the same folder as `results_<date-time>.zip`.

In [ ]:
# ---- settings ----
# Your Google Drive folder: the part after /folders/ in its link
# (https://drive.google.com/drive/folders/1EB43yD0e2c-kHmsbk46l89na6bvFMeMB).
# The two zips must be inside it; the results zip is saved back into it.
DRIVE_FOLDER_ID = "1EB43yD0e2c-kHmsbk46l89na6bvFMeMB"
WORK = "/content/safecrop-clip-finetune"

# What to run (one command per line, top to bottom).
COMMANDS = [
    "python scripts/check_setup.py",
    "python scripts/noisy_database.py --tag e1",                                   # E1
    # "python scripts/finetune.py --set finetune.folds=[0] --tag colab_quick",    # E3, fold 0 only
    # "python scripts/finetune.py --tag colab_full",                              # E3, all 5 folds
    # "python scripts/linear_probe.py --set linear_probe.classifier=svm --tag svm",
]

In [ ]:
# ---- is there a GPU? ----
import torch, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime/server to a T4 GPU")

In [ ]:
# ---- Google Drive: download the two zips from your folder (found by its ID) ----
# Works whether the folder is in your own Drive or shared with you: sign in with the account that can open it.
# If this cell fails (for example in VS Code), drag the two zips into the Files panel on the left
# (they then sit in /content) and run the next cells.
import io, os
drive_api = None
try:
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    drive_api = build("drive", "v3")
    found = drive_api.files().list(
        q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
        fields="files(id, name, size, modifiedTime)", orderBy="modifiedTime desc", pageSize=200,
        supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get("files", [])
    print("In the Drive folder:", [f["name"] for f in found] or "NOTHING (wrong folder ID or wrong Google account?)")
    for stem in ("safecrop_code", "safecrop_test_images_840"):
        match = [f for f in found if f["name"].startswith(stem) and f["name"].endswith(".zip")]
        if not match:
            print(f"MISSING: {stem}.zip is not in the Drive folder")
            continue
        f = match[0]                                        # the newest upload, if there are several
        print(f"downloading {f['name']} ({int(f.get('size', 0)) / 1e6:.0f} MB) ...", flush=True)
        with io.FileIO(f"/content/{stem}.zip", "wb") as fh:
            dl = MediaIoBaseDownload(fh, drive_api.files().get_media(fileId=f["id"], supportsAllDrives=True),
                                     chunksize=50 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
    print("Drive step finished")
except Exception as e:
    print("Could not use Google Drive here:", e)
    print("Drag the two zips into the Files panel on the left instead, then run the next cells.")

In [ ]:
# ---- unpack the code and the images ----
import zipfile, shutil, glob

def find(stem):
    """The newest zip in /content whose name starts with stem."""
    hits = sorted(glob.glob(f"/content/{stem}*.zip"), key=os.path.getmtime, reverse=True)
    if not hits:
        raise FileNotFoundError(f"{stem}.zip is not in /content: look at the Drive cell's output above")
    return hits[0]

keep = os.path.join(WORK, "results")
if os.path.isdir(keep):                                   # keep results of earlier runs in this session
    shutil.move(keep, "/content/_results_keep")
shutil.rmtree(WORK, ignore_errors=True)
with zipfile.ZipFile(find("safecrop_code")) as z:
    z.extractall("/content")                              # -> /content/safecrop-clip-finetune
with zipfile.ZipFile(find("safecrop_test_images_840")) as z:
    z.extractall(os.path.join(WORK, "data"))              # -> data/images/L0 ... and data/extra_references
if os.path.isdir("/content/_results_keep"):
    shutil.move("/content/_results_keep", keep)
print(len(glob.glob(f"{WORK}/data/images/L*/*/*.png")), "test images ready")

In [ ]:
# ---- packages (Colab already has torch and transformers) ----
!pip -q install pyyaml tqdm

In [ ]:
# ---- run (output appears below; a full 5-fold run takes ~10-20 min on a T4) ----
%cd {WORK}
import sys
ip = get_ipython()
for cmd in COMMANDS:
    print("\n$", cmd, flush=True)
    if cmd.startswith("python "):                        # use this notebook's own Python
        cmd = f'"{sys.executable}" ' + cmd[len("python "):]
    ip.system(cmd)
    if ip.user_ns.get("_exit_code", 0):
        print(f"stopped: '{cmd}' failed")
        break


In [ ]:
# ---- save the results into your Drive folder (as a zip, without the big embedding cache) ----
import datetime
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
out = f"/content/results_{stamp}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{WORK}/results/**/*", recursive=True):
        if os.path.isfile(p) and "/cache/" not in p:
            z.write(p, os.path.relpath(p, WORK))
try:
    from googleapiclient.http import MediaFileUpload
    saved = drive_api.files().create(
        body={"name": os.path.basename(out), "parents": [DRIVE_FOLDER_ID]},
        media_body=MediaFileUpload(out, mimetype="application/zip", resumable=True),
        fields="name", supportsAllDrives=True).execute()
    print("Saved to your Drive folder:", saved["name"])
except Exception as e:
    print("Could not save to Drive:", e)
    print("Downloading", out, "to your computer instead (it is also in the Files panel on the left)")
    from google.colab import files
    files.download(out)